# Missing Value Imputation Menggunakan WKNN

**1. Dataset Awal**

Dataset berikut berisi tiga atribut yaitu IPK, PO, dan JML. Pada data ke-7 terdapat nilai JML yang belum diketahui (missing value) sehingga perlu diprediksi menggunakan metode Weighted K-Nearest Neighbor (WKNN).

| No | IPK | PO     | JML |
| -- | --- | ------ | --- |
| 1  | 2   | 200000 | 2   |
| 2  | 3   | 300000 | 3   |
| 3  | 4   | 200000 | 2   |
| 4  | 2   | 200000 | 3   |
| 5  | 3   | 300000 | 2   |
| 6  | 4   | 400000 | 3   |
| 7  | 2   | 300000 | ?   |

Nilai JML pada data ke-7 tidak tersedia sehingga perlu dilakukan imputasi data dengan memanfaatkan kedekatan data lain yang memiliki karakteristik serupa.

# 2. Normalisasi Data Menggunakan Min-Max

Sebelum menghitung jarak antar data, atribut perlu dinormalisasi agar berada pada skala yang sama.

Rumus Min-Max Normalization:

$$
X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Keterangan:

 $$
\begin{aligned}
 -X &= \text{nilai asli} \\
 -X_{min} &= \text{nilai minimum pada atribut} \\
 -X_{max} &= \text{nilai maksimum pada atribut} \\
 -X_{norm} &= \text{hasil normalisasi}
\end{aligned}
$$

**Nilai Minimum dan Maksimum**

| Atribut | Minimum | Maksimum |
| ------- | ------- | -------- |
| IPK     | 2       | 4        |
| PO      | 200000  | 400000   |
| JML     | 2       | 3        |

 **Hasil Normalisasi**

 | IPK | PO  | JML |
| --- | --- | --- |
| 0   | 0   | 0   |
| 0.5 | 0.5 | 1   |
| 1   | 0   | 0   |
| 0   | 0   | 1   |
| 0.5 | 0.5 | 0   |
| 1   | 1   | 1   |
| 0   | 0.5 | ?   |


Data yang akan diprediksi (baris ke-7):

IPK = 0

PO = 0.5

# 3. Perhitungan Jarak Menggunakan Euclidean Distance

Untuk menentukan tetangga terdekat digunakan rumus Euclidean Distance.

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}
$$
**Perhitungan Jarak terhadap Data ke-7**

| Data | Perhitungan | Jarak | JML |
|------|-------------|-------|-----|
| 1 | $\sqrt{(0-0)^2 + (0-0.5)^2}$ | 0.5 | 2 |
| 2 | $\sqrt{(0.5-0)^2 + (0.5-0.5)^2}$ | 0.5 | 3 |
| 3 | $\sqrt{(1-0)^2 + (0-0.5)^2}$ | 1.118 | 2 |
| 4 | $\sqrt{(0-0)^2 + (0-0.5)^2}$ | 0.5 | 3 |
| 5 | $\sqrt{(0.5-0)^2 + (0.5-0.5)^2}$ | 0.5 | 2 |
| 6 | $\sqrt{(1-0)^2 + (1-0.5)^2}$ | 1.118 | 3 |

# 4. Menentukan K Tetangga Terdekat

Misalkan digunakan K = 3, maka dipilih tiga data dengan jarak terkecil.

| Data | Jarak | JML |
| ---- | ----- | --- |
| 1    | 0.5   | 2   |
| 2    | 0.5   | 3   |
| 4    | 0.5   | 3   |

5. Perhitungan Bobot WKNN

Pada metode Weighted KNN, setiap tetangga memiliki bobot berdasarkan jaraknya.

Rumus bobot:

$$
w = d_{1}
$$

Karena semua jarak bernilai 0.5, maka bobotnya sama.

| Data | Bobot | JML |
| ---- | ----- | --- |
| 1    | 2     | 2   |
| 2    | 2     | 3   |
| 4    | 2     | 3   |

Nilai prediksi dihitung menggunakan rata-rata berbobot.

$$
\frac{(2 \times 2) + (2 \times 3) + (2 \times 3)}{2 + 2 + 2}
=
\frac{16}{6}
=
2.67
$$

setelh dibulatkan menjadi:

**JML = 3**

# 6. Dataset Setelah Imputasi

| No | IPK | PO     | JML   |
| -- | --- | ------ | ----- |
| 1  | 2   | 200000 | 2     |
| 2  | 3   | 300000 | 3     |
| 3  | 4   | 200000 | 2     |
| 4  | 2   | 200000 | 3     |
| 5  | 3   | 300000 | 2     |
| 6  | 4   | 400000 | 3     |
| 7  | 2   | 300000 | **3** |

nilai yang hilang pada data ke-7 berhasil diperkirakan menggunakan metode WKNN.

# 7. Implementasi Menggunakan Python

Contoh implementasi perhitungan WKNN menggunakan Python dan library scikit-learn.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

# dataset
data = {
    'IPK':[2,3,4,2,3,4,2],
    'PO':[200000,300000,200000,200000,300000,400000,300000],
    'JML':[2,3,2,3,2,3,np.nan]
}

df = pd.DataFrame(data)

# normalisasi
scaler = MinMaxScaler()
df[['IPK','PO']] = scaler.fit_transform(df[['IPK','PO']])

# pisahkan data lengkap dan data missing
train = df[df['JML'].notna()]
test = df[df['JML'].isna()]

X_train = train[['IPK','PO']]
y_train = train['JML']

model = KNeighborsRegressor(n_neighbors=3, weights='distance')
model.fit(X_train, y_train)

pred = model.predict(test[['IPK','PO']])

print("Prediksi JML:", pred[0])

Prediksi JML: 2.3333333333333335


# 8. Penjelasan Perbedaan Hasil Manual dan Program

Hasil perhitungan manual menghasilkan nilai 3, sedangkan perhitungan menggunakan program dapat menghasilkan nilai yang sedikit berbeda.

Hal ini disebabkan karena pada dataset terdapat beberapa data yang memiliki jarak yang sama terhadap data yang diprediksi. Program akan memilih tetangga berdasarkan urutan tertentu pada dataset, sedangkan pada perhitungan manual kita memilih tetangga secara langsung.

Perbedaan pemilihan tetangga tersebut dapat menyebabkan nilai rata-rata yang diperoleh juga sedikit berbeda.